# GPT-3.5-Turbo on AQUA — Self-Consistency (k=40)

Runs `gpt-3.5-turbo` with self-consistency decoding (`k=40`) on the full AQUA-RAT test set.  
Uses the disk response cache so re-runs are free.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pipeline_io import load_openai_key_from_envfile
load_openai_key_from_envfile()

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set — add it to .env"

## 1. Load & preview the AQUA dataset

In [ ]:
from pipeline_data import load_benchmark_dataset
from prompts import BenchmarkType

dataset = load_benchmark_dataset(BenchmarkType.AQUA)
print(f"AQUA test set: {len(dataset)} questions")
print(f"Example: {dataset[0]['question'][:120]}...")
print(f"Gold answer: {dataset[0]['final_answer']}")

## 2. Cost estimate

In [ ]:
from batch_runner import estimate_cost

est = estimate_cost(dataset, BenchmarkType.AQUA, n=40)
for k, v in est.items():
    print(f"  {k}: {v}")

## 3. Run evaluation (real-time API with disk cache)

This calls the pipeline's `evaluate()` directly.  
Each response is cached to `response_cache/` — re-running is free.

In [ ]:
from pipeline_eval import evaluate

results = evaluate(
    model="gpt-3.5-turbo",
    benchmark=BenchmarkType.AQUA,
    dataset=dataset,
    method="self_consistency",
    k=40,
    self_consistency_temperature=0.7,
)

print(f"\n=== Results ===")
print(f"Model:    {results.model}")
print(f"Dataset:  {results.dataset}")
print(f"Method:   {results.method}")
print(f"k:        {results.k}")
print(f"Accuracy: {results.accuracy:.4f} ({results.correct}/{results.total})")
print(f"Parsed:   {results.parsed}/{results.total}")

## 4. Save results to CSV

In [ ]:
from pathlib import Path
from pipeline_io import write_results

output_path = Path("results/aqua_k40_gpt35.csv")
write_results([results], output_csv=output_path)
print(f"Saved to {output_path}")

## 5. Inspect failures

In [ ]:
failures = [d for d in results.details if not d["is_correct"]]
print(f"{len(failures)} incorrect out of {results.total}\n")

for f in failures[:5]:
    print(f"Q: {f['question'][:100]}...")
    print(f"  Gold: {f['gold_answer']}  |  Predicted: {f['predicted']}")
    print()

## (Alternative) Run via Batch API — 50% cheaper

Uncomment and use this instead of cell 4 if you want to save money.  
Turnaround is typically minutes for this dataset size.

In [ ]:
# from batch_runner import BatchPipeline
#
# bp = BatchPipeline()
#
# # Submit
# batch_id = bp.submit_batch(
#     model="gpt-3.5-turbo",
#     benchmark=BenchmarkType.AQUA,
#     dataset=dataset,
#     temperature=0.7,
#     n=40,
# )
#
# # Wait for completion
# bp.wait_for_batch(batch_id)
#
# # Download
# batch_results = bp.download_results(batch_id)
# print(f"Got responses for {len(batch_results)} questions")